### 测出这块卡的"天花板"

**目标**：得到四个参照数，后面所有模型数字都拿它们对照。

| 符号 | 含义 | 用在哪 |
|---|---|---|
| P_fp32 | fp32（TF32 关）matmul 平台吞吐 | fp32 基线的预期时间 |
| P_tf32 | fp32（TF32 开）matmul 平台吞吐 | 可选参考行 |
| P_bf16 | bf16 matmul 平台吞吐 | 混合精度的预期时间 |
| BW | 显存拷贝带宽 | optimizer step 等访存密集算子的预期 |

#### 0. 前置检查（共享纪律）

```bash
cd /data2/c220/hz/cs336
nvidia-smi          # 两卡显存应接近 0，且没有 vllm server
mkdir -p results
```

#### 1. 脚本 `scripts/matmul_peak.py`


#### 2. 运行

用 tmux 开两个窗格，一个监控，一个跑实验：

```bash
# 窗格 A：监控（-i 用的是物理卡号）
nvidia-smi -i 0 --query-gpu=timestamp,clocks.sm,power.draw,temperature.gpu,utilization.gpu \
  --format=csv -l 1 > results/clock_gpu0.csv

# 窗格 B：跑实验
CUDA_VISIBLE_DEVICES=0 uv run python scripts/matmul_peak.py --tag gpu0
```

跑完后停掉监控，再对 1 号卡重复一遍（监控里 `-i 1`，实验里 `CUDA_VISIBLE_DEVICES=1 ... --tag gpu1`）。

#### 3. 怎么读结果

- 看每种模式的 **n=8192 和 16384 的 median**。小 n 吞吐低是因为没喂满 GPU，平台值才是天花板。
- 预期顺序：bf16 > fp32(TF32 开) > fp32(TF32 关)。如果 TF32 开和关几乎一样，检查输出里 `float32_matmul_precision` 那行是否真的变了。
- 两张卡的平台值应接近。差距明显：检查有没有别的任务，或者 `clock_gpu*.csv` 里 SM 频率有没有掉。
- 平台值与 median 的差异：`max` 与 `median` 差距大，说明测量抖动或降频，以 median 为准并在表里注明。
- memcopy 的参照：规格书标称显存带宽约 896 GB/s，实测通常是它的一部分，以实测为准。

- 填好的表

| 项目 | 卡 0 | 卡 1 | 采用值 |
|---|---|---|---|
| P_fp32（TF32 关，n=8192 / 16384） | 32.84 / 33.04 | 33.69 / 33.93 | **33.4 TFLOPs/s** |
| P_tf32（TF32 开） | 48.84 / 49.74 | 48.72 / 49.57 | **49.2** |
| P_bf16 | 92.36 / 98.41 | 95.03 / 98.74 | **96.1**（±3%） |
| BW（memcopy，读+写） | 768.0 | 769.6 | **769 GB/s** |
| 功耗墙 | 300 W | 300 W | 300 W |
| 稳态 SM 频率（fp32 关 / TF32 开 / bf16） | 约 2540 / 2812 / 2795 MHz | 约2650/2800/2800 | 记录即可 |

倍数关系可以写进报告：bf16 / fp32 ≈ 2.9×，TF32 / fp32 ≈ 1.5×，bf16 / TF32 ≈ 2.0×。memcopy 769 GB/s 约为标称带宽 896 GB/s 的 86%（标称值请以官方规格为准）。

**可选的验证**：确认降频原因的话，可以在跑 fp32 阶段时执行 `nvidia-smi -q -d PERFORMANCE`，看 "Clocks Event Reasons" 里是否有 `SW Power Cap: Active`。

- 决定：fp32 基线 = TF32 关

后面所有"预期时间"使用：

| 精度 | P | 用于 |
|---|---|---|
| fp32 基线 | 33.4 TFLOPs/s | 3.2、3.3 的 fp32 部分 |
| bf16 autocast | 96.1 TFLOPs/s | 3.5 混合精度对比 |
| 访存 | 769 GB/s | optimizer step |

在 3.1 的"TF32 设置"栏填：`torch.set_float32_matmul_precision("highest")`（TF32 关），并在 `benchmark.py` 里显式设置。

- 立刻可以算出的预期值（Step 2 的落地）

用第 4 节的 FLOPs 除以上面的 P，**B=4，ctx=512，fp32，TF32 关**：

| 模型 | forward (ms) | backward (ms) | forward+backward (ms) |
|---|---|---|---|
| small | 16.0 | 32.0 | 48.0 |
| medium | 53.7 | 107.4 | 161.1 |
| large | 123 | 246 | 369 |

仅 forward 的大模型（B=4，ctx=512，no_grad）：

| 模型 | fp32 (ms) | bf16 (ms) |
|---|---|---|
| xl | 253 | 88 |
| 2.7B | 424（显存约 12.7 GiB，贴边，不建议） | 148 |

optimizer step 下限（28 B/参数 ÷ 769 GB/s）：small 约 4.7 ms，medium 约 15.4 ms。

**这些都是"理想上限"，也就是时间的下界**，实测应该更慢。原因是模型里的矩阵乘比 8192³ 小得多，效率达不到平台值。可以用你刚测的规模曲线估计：small 的 FFN matmul（2048×768 乘 768×3072）约 9.7 GFLOP，相当于 n≈1700 的方阵，对应曲线上 n=1024 到 2048 附近，fp32 约为平台值的 75% 到 90%，bf16 约为 80% 到 97%。所以：
- small 的 t_meas / t_pred 预期在 1.2 到 2 之间；
- 如果 small 在 ctx=128 时远高于 2，那是 kernel 发射开销，不是 matmul 慢。


## profile
### 1.用手算建立预期

一个训练步的 FLOPs 约为 **6 × 参数量（不含 embedding）× token 数**（forward 占 1/3，backward 占 2/3）。

例：small，batch=4，ctx=512。非 embedding 参数约 85M，token 数 = 2048，所以约 1 TFLOP/步。除以 Step 1 测出的实际吞吐，就是你"应该"看到的步时间。

实测和这个预期差 2 倍以内就算正常。差得很远，说明脚本有问题，或者被 launch 开销、非 matmul 算子拖慢，这正是后面要分析的内容。


#### 预测清单（

以下均为 small，$B=4$，$T=512$，fp32，单卡。

**(1) 阶段时间**

| 阶段 | FLOPs | $P$（TFLOP/s） | 理想时间 | 访存部分 | 预测总时间 |
|---|---|---|---|---|---|
| forward | 0.534 T | 33.4 | 16.0 ms | 约 20 ms | 约 37 ms |
| backward | 1.068 T | 33.4 | 32.0 ms | 取 forward 的约 2 倍作粗估 | 待估 |
| optimizer | 28 B/参数 | BW 769 GB/s | 4.7 ms（下限） | | 见下 |

optimizer 的 4.7 ms 是带宽下限。staff 的 AdamW 若是逐参数 Python 循环，实际会被 kernel 发射开销主导，预期明显高于下限（12.9 解释）。

**(2) 时间占比**（FLOPs 占比与预测时间占比并排）

| 项 | FLOPs 占比 | 预测时间占比 |
|---|---|---|
| 注意力（matmul + 缩放 + mask + softmax） | 7.2% | 约 35%（第 8 章） |
| FFN（GEMM + 逐元素） | 65.2% | 约 40% |
| 其余（投影、norm、RoPE、残差、`lm_head`、loss） | 约 27.6% | 约 25%（余量） |

**(3) forward kernel 数与时间，按类别**

| 类别 | 预测 kernel 数 | 预测时间占比 |
|---|---|---|
| gemm | 约 109（$(7L+1)$ 个线性层加 $2L$ 个注意力 matmul） | 约 45% |
| elementwise | 约 300 | 约 40% |
| reduce | 约 25 | 约 3% |
| softmax | 12 | 约 5% |
| copy / cat | 约 40 | 约 3% |

这张表是按第 8.9 的 kernel 清单粗略相加的，误差较大，实测后直接替换。

**(4) forward 时间，按区间**（沿用第 8.9、9.9）

| 区间 | 预测 kernel 数 | 预测时间 (ms) |
|---|---|---|
| embed | 1 | 0.02 |
| block（×12） | 约 600（每个约 50） | 34.8 |
| ln_final | 6 | 0.09 |
| lm_head | 1 | 0.94 |
| loss | 数个 | 1.1 |

##### 显存预估

| 部分 | 估算 |
|---|---|
| 权重 + 梯度 + Adam | $16P$ B（fp32，$P$ 为参数个数） |
| FFN 激活 | $5Nd_{ff}$ 每层 |
| 注意力分数 | $c\,BhT^2$ 每层，$c\approx1$ 到 3 |
| 其余 $N\times d$ 张量 | 约 15 到 25 份每层 |
| logits 与损失 | $NV$ 的约 3 到 4 份 |

```python
def est_peak_gib(P, d, L, h, V, B, T, c=2, other_per_layer=20):
    N = B * T
    static = 16 * P
    act = L * 4 * (5 * N * 4 * d + c * B * h * T * T + other_per_layer * N * d) + 4 * 4 * N * V
    return (static + act) / 2**30
```



### 2.`scripts/benchmark.py`测量

- 详见benchmark.ipynb


- 结果如下：

```bash
CUDA_VISIBLE_DEVICES=0 uv run python scripts/benchmark.py     --model small --ctx 512 --batch 4 --dtype float32 --mode all
device: NVIDIA GeForce RTX 5070 Ti  torch 2.11.0+cu130
model=small params=128.6M ctx=512 batch=4 dtype=float32 compile=False
forward FLOPs = 534.0 GFLOP  (step = 3x = 1.60 TFLOP)
mode        median      min      max  jitter     wall     tok/s  TFLOP/s  peak GiB   (ms)
fwd          37.72    37.40    37.76     1%    37.73     54275    14.16      0.71
fwdbwd      114.90   114.72   115.16     0%   114.71     17854    13.94      4.06
full        127.85   127.70   128.04     0%   127.72     16035    12.53      5.04
saved -> results/bench/bench_small_ctx512_b4_float32.json
```

### 3.nsys 测量

### 4.ncu测量